# 10 · Scale and speed (W4)

**Part A: more data.** At the natural fraud rate, XGBoost is trained on N = 1K, 2K, 5K, 10K, 20K, 50K, 100K and all 170,883 training rows (nested subsets, the 10K one is exactly the uniform 10K context). The foundation models are drawn on the same axis with their 1K-10K contexts, and TabICLv2 also with 20K and 50K context rows (TabPFN v2 stops at 10K). Question: does the gap to XGBoost come from the context being capped?

**Part B: speed.** Time to score 1, 100 and 1,000 transactions, for XGBoost on GPU and on CPU and for both foundation models with a 1K and a 10K context. A card authorisation can wait a few milliseconds.

**Runtime (T4 GPU):** XGBoost curve about 15 minutes, TabICLv2 at 20K and 50K about 45 minutes, timing about 15 minutes.

In [ ]:
# 1. What to run.
SEEDS = [0, 1, 2, 3, 4]
SIZES = [1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 170_883]
RUN_LARGE_CONTEXTS = True    # TabICLv2 with 20K and 50K context rows
LARGE_K = [20_000, 50_000]

In [ ]:
# 2. Drive, latest code, libraries. Needs a GPU runtime.
from google.colab import drive
drive.mount("/content/drive")
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# 3. Tests, data, and the split fingerprint check.
!cd {REPO_DIR} && PYTHONPATH=src python -m pytest tests -q
import json
from fraudicl.config import DRIVE_DIR
from fraudicl.data import load_ulb, features_and_target, load_splits, fingerprint
X, y = features_and_target(load_ulb())
splits = load_splits(f"{DRIVE_DIR}/splits/ulb_splits.npz")
expected = open(f"{REPO_DIR}/results/ulb_splits_fingerprint.txt").readline().strip()
assert fingerprint(splits) == expected, "splits on Drive differ from the ones on record"
OUT = f"{DRIVE_DIR}/results"
by_key = {(s.seed, s.level): s for s in splits}
# the settings tuned on the full natural-rate training set in notebook 02
best = json.load(open(f"{OUT}/xgb_params/plain__natural.json"))
print("splits OK", expected, "| XGBoost settings:", best["params"])

In [ ]:
# 4. Part A: XGBoost on N rows, then TabICLv2 with 20K and 50K context rows.
from fraudicl.scale import run_xgb_scale
from fraudicl.experiments import run_tfm_sweep
from fraudicl.models import make_tabicl_v2, make_tabpfn_v2
run_xgb_scale(X, y, splits, OUT, SIZES, SEEDS, best["params"], level="natural")
if RUN_LARGE_CONTEXTS:
    run_tfm_sweep(X, y, splits, OUT, {"TabICLv2": make_tabicl_v2}, levels=["natural"],
                  budgets=LARGE_K, seeds=SEEDS, policy="uniform_large")

In [ ]:
# 5. Part A: the table and Figure 7.
from fraudicl import report as R
from fraudicl.scale import scale_table, plot_scale
from IPython.display import Image, display
runs = R.load_runs(OUT)
scale = scale_table(runs, level="natural")
full = runs[(runs.model == R.HEADLINE_XGB) & (runs.level.astype(str) == "natural")].pr_auc.mean()
os.makedirs("/content/export_10/figures", exist_ok=True)
plot_scale(scale, "/content/export_10/figures/fig7_scale.png", full_xgb=full)
display(Image("/content/export_10/figures/fig7_scale.png"))
scale.round(4)

**How to read Part A.** Read across at the same number of rows: XGBoost on N rows against a foundation model with N context rows. If the TabICLv2 line keeps rising at 20K and 50K and approaches the dotted full-data line, the gap at 0.172% is mostly the context cap. If it flattens while XGBoost keeps climbing, more rows alone won't close it, and the class-balanced contexts (W1) are the better route.

In [ ]:
# 6. Part B: speed. One model of each kind, fitted on seed 0 at the natural rate.
from fraudicl.scale import latency_table, scale_rows
from fraudicl.xgb import fit_xgb
s = by_key[0, "natural"]
models = {}
xgb_gpu = fit_xgb(X[s.train], y[s.train], X[s.val], y[s.val], best["params"], 0)
models["XGBoost, all rows (GPU)"] = xgb_gpu
xgb_cpu = fit_xgb(X[s.train], y[s.train], X[s.val], y[s.val], best["params"], 0)
xgb_cpu.set_params(device="cpu")
models["XGBoost, all rows (CPU)"] = xgb_cpu
for k in (1_000, 10_000):
    ctx = scale_rows(s.train, k, 0)          # the uniform context of size k
    for name, make in (("TabICLv2", make_tabicl_v2), ("TabPFN v2", make_tabpfn_v2)):
        m = make(0)
        m.fit(X[ctx], y[ctx])
        models[f"{name}, {k // 1000}K context (GPU)"] = m
latency = latency_table(models, X[s.test], batch_sizes=(1, 100, 1_000), repeats=3)
latency.round(4)

**How to read Part B.** `ms_per_transaction` at batch size 1 is the latency of scoring a single card payment as it happens. The foundation models process their whole context on every call, so one transaction costs about as much as the context; batching spreads that cost. XGBoost only walks its trees, so it stays far below a millisecond per transaction.

In [ ]:
# 7. Export: print as CSV text (copied into results/ on the laptop) and save on Drive.
EXPORT = f"{OUT}/export_10"
os.makedirs(EXPORT, exist_ok=True)
scale.to_csv(f"{EXPORT}/scale_natural.csv", index=False)
latency.to_csv(f"{EXPORT}/latency.csv", index=False)
print("=====SCALE====="); print(scale.to_csv(index=False))
print("=====LATENCY====="); print(latency.to_csv(index=False))
print("=====FULLXGB=====", full)